# Signal Hunters - Team Cyclops: final stack

CPU only, no internet. Inputs: the Round 2 data, the output of the main notebook (features,
LightGBM, CNNs, EfficientNets, repo code) and the output of the extra-models notebook. All base
models' out-of-fold and test probabilities are stacked by the same second-level LightGBM, then the
per-class bias is applied. Final file: `submissions/22_stackall_bias.csv`.

In [ ]:
import glob, os, shutil, subprocess
hits = glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)
os.environ['SIG_DATA'] = os.path.dirname(hits[0])
roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/oof/lgbm4_oof.csv', recursive=True)]
assert roots, 'Add the main notebook output as an input'
main = roots[0]
shutil.copytree(main, '/kaggle/working/signal-hunters', dirs_exist_ok=True)
os.environ['SIG_FEATURES'] = glob.glob('/kaggle/input/**/features/train_deg.parquet', recursive=True)[0].rsplit('/', 1)[0]
for p in glob.glob('/kaggle/input/**/oof/*.csv', recursive=True):
    dst = '/kaggle/working/signal-hunters/oof/' + os.path.basename(p)
    if not os.path.exists(dst):
        shutil.copy(p, dst)
        print('added', os.path.basename(p))
os.chdir('/kaggle/working/signal-hunters')
print(main, os.environ['SIG_FEATURES'])

## Stack every base model, then the per-class bias

In [ ]:
models = [m for m in ['lgbm4', 'cnn', 'cnn_s7', 'cnn_lvl', 'eff_lvl', 'effstk_lvl', 'effb2_lvl', 'eff_lvl_e20',
                     'eff_lvl_s11', 'effstk_lvl_s7', 'res34_lvl']
          if os.path.exists(f'oof/{m}_oof.csv') and os.path.exists(f'oof/{m}_test.csv')]
print('stacking', models)
subprocess.run(['python', 'scripts/stack.py', '--models', *models, '--features', 'fit_', 'trk_', 'lvl_',
                '--name', 'stackall'], check=True)
!python scripts/make_submission.py --models stackall --bias --out 22_stackall_bias
!ls -la submissions